# **Graph Knowledge Final Project**
*   Khalila Shafarayhani Atletiko - 5026231167
*   Alisha Rafimalia - 5026231202

### **Instalasi Library**
Perintah ini mengunduh dan memasang modul-modul dari ekosistem LangChain—yaitu sebuah framework yang berfungsi untuk memudahkan integrasi antara model kecerdasan buatan (LLM) dengan sumber data eksternal seperti database.

In [1]:
!pip install langchain langchain-openai langchain-neo4j langchain-experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.2/58.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.3/262.3 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not cur

### **Import Konfigurasi dan Lingkungan**
Mengatur variabel lingkungan (kredensial) agar Python mendapatkan izin akses masuk ke dalam database Neo4j serta akses untuk menggunakan layanan AI dari OpenRouter menggunakan API Key.

In [2]:
import os
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_openai import ChatOpenAI
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_core.documents import Document

# 1. Kredensial Database Neo4j
os.environ["NEO4J_URI"] = "bolt://3.237.72.2:7687"
os.environ["NEO4J_USERNAME"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = "chit-career-polishes"

# 2. Kredensial OpenRouter
os.environ["OPENROUTER_API_KEY"] = "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"

/tmp/ipykernel_5182/4066658291.py:4: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


### **Inisialisasi Koneksi Database dan LLM**

In [3]:
print("Menghubungkan ke Neo4j...")
graph = Neo4jGraph()
graph.refresh_schema()
print("Skema Graf berhasil dimuat!")

print("Menginisialisasi LLM dari OpenRouter...")
llm = ChatOpenAI(
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    model_name="qwen/qwen-2.5-72b-instruct", # Kamu bisa ganti ke model Llama jika mau
    temperature=0,
    model_kwargs={
        "extra_headers": {
            "HTTP-Referer": "https://github.com/khalilash",
            "X-Title": "Tugas Geopolitik Neo4j"
        }
    }
)
print("Koneksi LLM Siap!")

Menghubungkan ke Neo4j...
Skema Graf berhasil dimuat!
Menginisialisasi LLM dari OpenRouter...
Koneksi LLM Siap!


### **Komponen Text to Cypher & Graph RAG**

In [4]:
print("--- Eksekusi Text-to-Cypher & Graph RAG ---")

# Setup Chain
cypher_rag_chain = GraphCypherQAChain.from_llm(
    graph=graph,
    cypher_llm=llm,
    qa_llm=llm,
    verbose=True,
    return_intermediate_steps=True,
    allow_dangerous_requests=True
)

# Pertanyaan uji coba
pertanyaan = "Ada berapa kota di negara Indonesia berdasarkan database, dan bahasa apa yang digunakan di negara tersebut?"

try:
    response = cypher_rag_chain.invoke({"query": pertanyaan})
    print("\n[HASIL TEXT-TO-CYPHER]")
    print(response["intermediate_steps"][0]["query"])

    print("\n[HASIL GRAPH RAG (Jawaban Natural)]")
    print(response["result"])
except Exception as e:
    print("Terjadi kesalahan:", e)

--- Eksekusi Text-to-Cypher & Graph RAG ---


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:City)-[:BELONGS_TO]->(country:Country {name: 'Indonesia'})-[:SPEAKS]->(language:Language)
RETURN count(DISTINCT c), collect(DISTINCT language.name)

Full Context:
[{'count(DISTINCT c)': 13, 'collect(DISTINCT language.name)': ['Indonesian']}]

> Finished chain.

[HASIL TEXT-TO-CYPHER]
MATCH (c:City)-[:BELONGS_TO]->(country:Country {name: 'Indonesia'})-[:SPEAKS]->(language:Language)
RETURN count(DISTINCT c), collect(DISTINCT language.name)


[HASIL GRAPH RAG (Jawaban Natural)]
Berdasarkan database, terdapat 13 kota di negara Indonesia, dan bahasa yang digunakan di negara tersebut adalah Bahasa Indonesia.


### **Komponen LLM Graph Builder**

In [5]:
print("--- Eksekusi LLM Graph Builder ---")

# Teks berita simulasi (Unstructured Data)
teks_berita_unstructured = """
Indonesia baru-baru ini mempererat kerja sama ekonomi dengan organisasi BRICS.
Selain itu, Jakarta sebagai ibu kota Indonesia menjadi pusat diplomasi untuk wilayah Asia Tenggara.
Negara ini secara aktif berdagang dengan China menggunakan mata uang Yuan dalam beberapa transaksi bilateral.
"""

documents = [Document(page_content=teks_berita_unstructured)]

# Setup Transformer dengan batasan skema agar rapi
llm_graph_builder = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["Country", "City", "Organization", "Currency", "Region"],
    allowed_relationships=["MEMBER_OF", "CAPITAL_OF", "LOCATED_IN", "TRADES_WITH", "USES_CURRENCY"]
)

print("Mengekstrak entitas dan relasi dari teks (Mohon tunggu beberapa detik)...")
graph_documents = llm_graph_builder.convert_to_graph_documents(documents)

# Menampilkan hasil ekstraksi LLM
for i, doc in enumerate(graph_documents):
    print(f"\n[HASIL EKSTRAKSI NODE]")
    for node in doc.nodes:
        print(f" - {node.id} ({node.type})")

    print(f"\n[HASIL EKSTRAKSI RELASI]")
    for rel in doc.relationships:
        print(f" - ({rel.source.id}) -[{rel.type}]-> ({rel.target.id})")

# Memasukkan hasil ekstraksi ke dalam database Neo4j
graph.add_graph_documents(graph_documents)
print("\n[STATUS] Sukses! Data dari teks unstructured berhasil di-inject ke database Neo4j.")

--- Eksekusi LLM Graph Builder ---
Mengekstrak entitas dan relasi dari teks (Mohon tunggu beberapa detik)...

[HASIL EKSTRAKSI NODE]
 - Indonesia (Country)
 - Brics (Organization)
 - Jakarta (City)
 - Asia Tenggara (Region)
 - China (Country)
 - Yuan (Currency)

[HASIL EKSTRAKSI RELASI]
 - (Indonesia) -[MEMBER_OF]-> (Brics)
 - (Jakarta) -[CAPITAL_OF]-> (Indonesia)
 - (Jakarta) -[LOCATED_IN]-> (Asia Tenggara)
 - (Indonesia) -[TRADES_WITH]-> (China)
 - (Indonesia) -[USES_CURRENCY]-> (Yuan)

[STATUS] Sukses! Data dari teks unstructured berhasil di-inject ke database Neo4j.
